### 2.3. Load dataset
In this final part of data cleaning, we will follow the same cleaning framework as the former.

But before executing the Market Time Unit harmonization, this process includes the creation of two datasets:  
I. The first one will be used for national comparison to the Supply and Prices data;  
II. The second one, is created for unnecessary analysis that could still provide valuable insights on zonal consumption trends.

In [3]:
# import libraries
import pandas as pd
from pathlib import Path
import warnings

# filter User and Future Warnings for privacy matters
warnings.simplefilter(action="ignore", category=UserWarning)
warnings.simplefilter(action="ignore", category=FutureWarning)

# define initial variables
dataset_path = Path('../dataset/raw')           # input folder path
output_path = Path('../dataset/clean/updated')  # output folder path
output_path.mkdir(parents=True, exist_ok=True)  # create folder if non-existent

YEARS = range(2023, 2027)

In [4]:
# 1. Aggregate load data in dataframe
load_dfs = [
    pd.read_excel(dataset_path / f'TERNA-load-{year}.xlsx')
    for year in YEARS
]
load_data = pd.concat(load_dfs, ignore_index=True)

# fast check
print(f'''
----Rows x Cols:----
{load_data.shape}\n
----First rows:----
{load_data.head()}\n
----Last rows:----
{load_data.tail()}
''')


----Rows x Cols:----
(977631, 4)

----First rows:----
                  Date  Total Load [MW]  Forecast Total Load [MW]  \
0  2023-12-31 23:45:00          565.863                   560.580   
1  2023-12-31 23:45:00         1998.771                  1980.110   
2  2023-12-31 23:45:00         4663.746                  4620.205   
3  2023-12-31 23:45:00        11502.558                 11395.170   
4  2023-12-31 23:45:00          823.763                   816.072   

   Bidding Zone  
0      Calabria  
1  Centre-North  
2  Centre-South  
3         North  
4      Sardinia  

----Last rows:----
                                   Date  Total Load [MW]  \
977626              2026-01-01 00:00:00         4569.799   
977627              2026-01-01 00:00:00         2292.859   
977628              2026-01-01 00:00:00          752.189   
977629              2026-01-01 00:00:00        26098.002   
977630  Applied filters:  Year is 2026;              NaN   

        Forecast Total Load [MW]  Bidding

In [5]:
# check
print(f'''
{load_data.info()}\n
----Date description----
{load_data['Date'].describe()}\n
----Actual energy load description----
{load_data['Total Load [MW]'].describe()}\n
----Forecast energy load description----
{load_data['Forecast Total Load [MW]'].describe()}\n
----Source of production----
{load_data['Bidding Zone'].describe()}\n
''')

<class 'pandas.DataFrame'>
RangeIndex: 977631 entries, 0 to 977630
Data columns (total 4 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   Date                      977631 non-null  object 
 1   Total Load [MW]           977627 non-null  float64
 2   Forecast Total Load [MW]  977627 non-null  float64
 3   Bidding Zone              977627 non-null  str    
dtypes: float64(2), object(1), str(1)
memory usage: 29.8+ MB

None

----Date description----
count                  977631
unique                 122196
top       2023-10-29 02:45:00
freq                       16
Name: Date, dtype: object

----Actual energy load description----
count    977627.000000
mean       8843.666571
std       12105.505301
min         -74.350000
25%        1354.219000
50%        2818.703000
75%        9843.634500
max       98216.000000
Name: Total Load [MW], dtype: float64

----Forecast energy load description----
count    977627.000000

In [6]:
# 2. Delete n=5 Rows with NULL values it's the row at the end of each dataset (for each year) which explains the applied filter
load_data = load_data.dropna(axis=0, how='any')
load_data.head(10)       # check

,Date,Total Load [MW],Forecast Total Load [MW],Bidding Zone
0,2023-12-31 23:45:00,565.863,560.580,Calabria
1,2023-12-31 23:45:00,1998.771,1980.110,Centre-North
2,2023-12-31 23:45:00,4663.746,4620.205,Centre-South
3,2023-12-31 23:45:00,11502.558,11395.170,North
4,2023-12-31 23:45:00,823.763,816.072,Sardinia
5,2023-12-31 23:45:00,1773.253,1756.698,Sicily
6,2023-12-31 23:45:00,1594.046,1579.164,South
7,2023-12-31 23:45:00,22922.000,22707.999,Italy
8,2023-12-31 23:30:00,23349.999,23027.001,Italy
9,2023-12-31 23:30:00,570.593,562.700,Calabria


In [7]:
# 3. Convert Date from Object type to Datetime type
load_data['Date'] = pd.to_datetime(load_data['Date'], dayfirst=True, errors='coerce')
#    Convert Bidding Zone from Object to String
load_data['Bidding Zone'] = load_data['Bidding Zone'].astype('string')

From this dataset, it is helpful to create two separate tables:
1. The first one only selects the Italy column which is already provided by TERNA and it serves for analyzing the total load in Italy;
2. The second one, which details the bidding zone, could be helpful for zonal analysis.

This data separation has been implemented for optimizing hard disk space and time analysis, since the merged dataset is more than 1mln rows long.

In [8]:
# 1. Italy dataset
load_italy = load_data[load_data['Bidding Zone'] == 'Italy'].copy()

load_italy = (
    load_italy
    .groupby(pd.Grouper(key='Date', freq='h'))[['Total Load [MW]', 'Forecast Total Load [MW]']]
    .mean()
    .reset_index()
)

load_italy['Total Load [MW]'] = load_italy['Total Load [MW]'].clip(lower=0)

# rename columns
load_italy = load_italy.rename(columns={
    'Total Load [MW]': 'Total Load',
    'Forecast Total Load [MW]': 'Load forecasted'
})

load_italy

,Date,Total Load,Load forecasted
0,2023-01-01 00:00:00,21644.24950,22366.75050
1,2023-01-01 01:00:00,20891.00000,20775.74975
2,2023-01-01 02:00:00,19762.24925,19644.24900
3,2023-01-01 03:00:00,18765.50050,18837.25050
4,2023-01-01 04:00:00,18073.75000,18715.25000
...,...,...,...
30595,2026-06-28 19:00:00,43487.25000,42911.50000
30596,2026-06-28 20:00:00,43231.00000,42299.00000
30597,2026-06-28 21:00:00,43050.25100,42659.74975
30598,2026-06-28 22:00:00,41414.99975,41477.75050


In [9]:
# 2. Zonal dataset
load_zones = load_data[load_data['Bidding Zone'] != 'Italy'].copy()

load_zones = (
    load_zones
    .groupby(['Bidding Zone', pd.Grouper(key='Date', freq='h')])['Total Load [MW]']
    .mean()
    .reset_index()
)

load_zones_pivot = load_zones.pivot(
    index='Date',
    columns='Bidding Zone',
    values='Total Load [MW]'
)

load_zones_pivot

Bidding Zone,Calabria,Centre-North,Centre-South,North,Sardinia,Sicily,South
Date,,,,,,,
2023-01-01 00:00:00,552.02950,1820.29225,4379.11850,10916.33775,806.13675,1659.60925,1510.72550
2023-01-01 01:00:00,525.32875,1750.74675,4209.03475,10599.71175,783.55525,1576.97350,1445.64925
2023-01-01 02:00:00,463.02800,1674.28550,3971.24375,9982.60200,753.62675,1581.43925,1336.02400
2023-01-01 03:00:00,423.18575,1616.10000,3777.64725,9491.62225,736.34950,1502.49325,1218.10250
2023-01-01 04:00:00,420.02650,1532.98125,3691.74000,9166.81275,723.29950,1449.77500,1089.11500
...,...,...,...,...,...,...,...
2026-06-28 19:00:00,1071.03325,3503.46725,7100.19950,24683.13325,1333.27225,2768.80075,3027.34375
2026-06-28 20:00:00,1092.89275,3457.44000,7147.39250,24262.33825,1352.97725,2828.37575,3089.58350
2026-06-28 21:00:00,1084.76475,3471.14700,7242.29700,24064.16600,1334.39300,2911.17050,2942.31275


After creating two separate datasets for two different aims, data exportation of both datasets into the output folder is executed.

In [10]:
# export new datasets
load_italy.to_csv(output_path / 'load-italy-clean.csv', index=False)
load_zones_pivot.to_csv(output_path / 'load-zones-clean.csv')